# Model Experiments

In [34]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

In [23]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('Online_Shoppers_Base')

<Experiment: artifact_location='/home/rabat/projects/behavior-engineering/artifacts/1', creation_time=1790795785035, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790795785035, lifecycle_stage='active', name='Online_Shoppers_Base', tags={}, trace_location=None, workspace='default'>

In [24]:
df = pd.read_csv('../data/online_shoppers_intention.csv', sep=',')
df.head()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [25]:
target_col = 'Revenue'
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [26]:
colunas_para_dropar = ['Weekend', 'Region', 'Browser', 'OperatingSystems']
colunas_categoricas = ['Month', 'TrafficType', 'VisitorType']

In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('drop_ruido', 'drop', colunas_para_dropar),
        ('ohe', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=0.01), colunas_categoricas)
    ],
    remainder='passthrough'
)

In [28]:
modelos = {
    "Decision_Tree": DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
    "Random_Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, scale_pos_weight=5.45, eval_metric="aucpr", random_state=42)
}

In [29]:
for nome, modelo in modelos.items():
    with mlflow.start_run(run_name=f"Pipeline_{nome}", nested=True):
        
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', modelo)
        ])
        
        print(f"Treinando {nome}...\n")
        pipeline.fit(X_train, y_train)
        
        y_pred = pipeline.predict(X_test)
        y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
    
        accuracy = float(accuracy_score(y_test, y_pred))
        mlflow.log_metric('test_accuracy', accuracy)
        
        f1 = float(f1_score(y_test, y_pred))
        mlflow.log_metric('test_f1', f1)
        
        precision = float(precision_score(y_test, y_pred))
        mlflow.log_metric('test_precision', precision)
        
        recall = float(recall_score(y_test, y_pred))
        mlflow.log_metric('test_recall', recall)
    
        pr_auc = float(average_precision_score(y_test, y_pred_proba))
        mlflow.log_metric('test_pr_auc', pr_auc)

    print(
    f'\nTreino do modelo {nome} concluído.\n'
    f'PR-AUC: {pr_auc:.2f}\n'
    f'Accuracy: {accuracy:.2f}\n'
    f'Precision: {precision:.2f}\n'
    f'Recall: {recall:.2f}\n'
    f'F1: {f1:.2f}\n'
)

Treinando Decision_Tree...

🏃 View run Pipeline_Decision_Tree at: http://127.0.0.1:5000/#/experiments/1/runs/219b3598cf824e63924199418cdbc28f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Treino do modelo Decision_Tree concluído.
PR-AUC: 0.64
Accuracy: 0.83
Precision: 0.48
Recall: 0.85
F1: 0.62

Treinando Random_Forest...

🏃 View run Pipeline_Random_Forest at: http://127.0.0.1:5000/#/experiments/1/runs/80df4f6327434792a26fe6982a9f1389
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Treino do modelo Random_Forest concluído.
PR-AUC: 0.71
Accuracy: 0.88
Precision: 0.61
Recall: 0.69
F1: 0.65

Treinando XGBoost...

🏃 View run Pipeline_XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/948deda4c00644868a2c288f4f0f13e6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Treino do modelo XGBoost concluído.
PR-AUC: 0.71
Accuracy: 0.88
Precision: 0.59
Recall: 0.71
F1: 0.64



In [30]:
def objective(trial):
    # MLflow Nested Run para cada tentativa
    with mlflow.start_run(nested=True, run_name=f"Trial_{trial.number}"):
        
        # Espaço de busca focado na arquitetura do XGBoost
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "scale_pos_weight": 5.45,
            "eval_metric": "aucpr",
            "random_state": 42,
            "tree_method": "hist",
            "device": "cuda" # Troque para "cpu" se for rodar na VPS / ambiente sem GPU
        }
        
        # Loga os parâmetros testados no MLflow
        mlflow.log_params(params)
        
        # StratifiedKFold: 5 recortes mantendo a proporção 85/15
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        pr_aucs = []
        accs = []
        prcs = []
        recs = []
        f1s = []

        
        # O Cross-Validation acontece aqui dentro
        for train_idx, val_idx in skf.split(X_train, y_train):
            # Recorta os dados
            X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            # Monta e treina o pipeline para EVITAR DATA LEAKAGE no fold
            pipeline = Pipeline(steps=[
                ('preprocessor', preprocessor),
                ('classifier', XGBClassifier(**params))
            ])
            
            pipeline.fit(X_fold_train, y_fold_train)
            
            # Avalia
            y_pred = pipeline.predict(X_fold_val)
            y_prob = pipeline.predict_proba(X_fold_val)[:, 1]

            fold_accuracy = accuracy_score(y_fold_val, y_pred)
            fold_precision = precision_score(y_fold_val, y_pred)
            fold_recall = recall_score(y_fold_val, y_pred)
            fold_f1 = f1_score(y_fold_val, y_pred)
            fold_pr_auc = average_precision_score(y_fold_val, y_prob)

            pr_aucs.append(fold_pr_auc)
            accs.append(fold_accuracy)
            prcs.append(fold_precision)
            recs.append(fold_recall)
            f1s.append(fold_f1)
            
        # O veredito dessa tentativa é a média dos 5 folds
        mean_acc = np.mean(accs)
        mean_precision = np.mean(prcs)
        mean_recall = np.mean(recs)
        mean_f1 = np.mean(f1s)
        mean_pr_auc = np.mean(pr_aucs)
        
        mlflow.log_metric('cv_mean_acc', mean_acc)
        mlflow.log_metric('cv_mean_precision', mean_precision)
        mlflow.log_metric('cv_mean_recall', mean_recall)
        mlflow.log_metric('cv_mean_f1', mean_f1)
        mlflow.log_metric("cv_mean_pr_auc", mean_pr_auc)

        
        return mean_pr_auc

In [31]:
with mlflow.start_run(run_name="Optuna_XGBoost", nested=True):
    # Cria o estudo mirando em MAXIMIZAR o PR-AUC
    study = optuna.create_study(direction="maximize")
    # 30 trials é um bom começo. Aumente se quiser torrar hardware.
    study.optimize(objective, n_trials=30)
    
    print(f"\nMelhor PR-AUC no CV: {study.best_value:.4f}")
    print("Melhores hiperparâmetros:", study.best_params)

[I 2026-10-01 12:52:18,253] A new study created in memory with name: no-name-72650f81-5f28-470b-bb5a-39c80b39ddb6
[I 2026-10-01 12:52:20,047] Trial 0 finished with value: 0.7092461539191834 and parameters: {'n_estimators': 177, 'learning_rate': 0.1743392046655828, 'max_depth': 7, 'subsample': 0.6078153657002151, 'colsample_bytree': 0.7097345040721614}. Best is trial 0 with value: 0.7092461539191834.


🏃 View run Trial_0 at: http://127.0.0.1:5000/#/experiments/1/runs/d36c0348619a4ec4b897b5976ed0a2e8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-10-01 12:52:21,564] Trial 1 finished with value: 0.7482696320750752 and parameters: {'n_estimators': 286, 'learning_rate': 0.033519590508367524, 'max_depth': 4, 'subsample': 0.6268024484424537, 'colsample_bytree': 0.6274794998230437}. Best is trial 1 with value: 0.7482696320750752.


🏃 View run Trial_1 at: http://127.0.0.1:5000/#/experiments/1/runs/27c93e5d9b174948af840211d3ab4ca6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-10-01 12:52:24,266] Trial 2 finished with value: 0.7256471165995679 and parameters: {'n_estimators': 236, 'learning_rate': 0.0709119210845429, 'max_depth': 9, 'subsample': 0.6663083329952538, 'colsample_bytree': 0.6926095011453012}. Best is trial 1 with value: 0.7482696320750752.


...

🏃 View run Trial_27 at: http://127.0.0.1:5000/#/experiments/1/runs/4deae77457fd4925ad82e84288c124cb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-10-01 12:53:00,390] Trial 28 finished with value: 0.7472921304805024 and parameters: {'n_estimators': 236, 'learning_rate': 0.03500919966615779, 'max_depth': 5, 'subsample': 0.8616726892294669, 'colsample_bytree': 0.9355641610562733}. Best is trial 12 with value: 0.7518715725555071.


🏃 View run Trial_28 at: http://127.0.0.1:5000/#/experiments/1/runs/33e8520e6d8c4960ab96cd6e3075c61c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-10-01 12:53:01,371] Trial 29 finished with value: 0.7482305268703836 and parameters: {'n_estimators': 69, 'learning_rate': 0.06414932009992556, 'max_depth': 5, 'subsample': 0.7785071102069304, 'colsample_bytree': 0.9010014246500361}. Best is trial 12 with value: 0.7518715725555071.


🏃 View run Trial_29 at: http://127.0.0.1:5000/#/experiments/1/runs/ea8d266033b748a8a3358526e2e557ab
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Melhor PR-AUC no CV: 0.7519
Melhores hiperparâmetros: {'n_estimators': 177, 'learning_rate': 0.02929080317644553, 'max_depth': 4, 'subsample': 0.8134838044580179, 'colsample_bytree': 0.956810393362548}
🏃 View run Optuna_XGBoost at: http://127.0.0.1:5000/#/experiments/1/runs/af7dabb4bb904976b389cd6e07677471
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [32]:
with mlflow.start_run(run_name='XGBoost_Champion'):
    # 5. Treino do Modelo Campeão no X_train INTEIRO
    print("\nTreinando o modelo campeão com todos os dados de treino...")
    melhores_params = study.best_params
    melhores_params.update({"scale_pos_weight": 5.45, "eval_metric": "aucpr", "random_state": 42, "tree_method": "hist", "device": "cuda"})
    
    campeao_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', XGBClassifier(**melhores_params))
    ])
    
    campeao_pipeline.fit(X_train, y_train)
    
    # Avaliação final no conjunto de TESTE INTACTO
    y_pred = campeao_pipeline.predict(X_test)
    y_pred_proba = campeao_pipeline.predict_proba(X_test)[:, 1]

    accuracy = float(accuracy_score(y_test, y_pred))
    f1 = float(f1_score(y_test, y_pred))
    precision = float(precision_score(y_test, y_pred))
    recall = float(recall_score(y_test, y_pred))
    pr_auc = average_precision_score(y_test, y_pred_proba)
    
    mlflow.log_metric('test_accuracy', accuracy)
    mlflow.log_metric('test_f1', f1)
    mlflow.log_metric('test_precision', precision)
    mlflow.log_metric('test_recall', recall)
    mlflow.log_metric("test_pr_auc", float(pr_auc))

    print(f"PR-AUC FINAL NO TESTE (A prova de fogo): {pr_auc:.4f}")


Treinando o modelo campeão com todos os dados de treino...
PR-AUC FINAL NO TESTE (A prova de fogo): 0.7515
🏃 View run XGBoost_Champion at: http://127.0.0.1:5000/#/experiments/1/runs/57b5b7b58dca4033ab49d619d55013f4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [35]:
joblib.dump(campeao_pipeline, '../models/champion_model.joblib')

['../models/champion_model.joblib']